In [32]:
import numpy as np
import pandas as pd
# from sklearn.preprocessing import StandardScaler

In [4]:
columns = ['size', 'est_price', 'rooms', 'price']

train_data = [
	[96.0,	900_000,	2,	1_200_000],
	[118.0,	1_250_000,	2,	1_900_000],
	[141.2,	1_500_000,	3,	3_000_000],
	[129.6,	1_450_000,	3,	2_800_000],
	[110.0,	1_150_000,	2,	1_600_000],
	[105.0,	1_000_000,	2,	1_500_000],
]

test_data = [
	[125.0,	1_300_000,	3,	None]
]

In [5]:
train_df = pd.DataFrame(data=train_data, columns=columns)
train_df

,size,est_price,rooms,price
0,96.0,900000,2,1200000
1,118.0,1250000,2,1900000
2,141.2,1500000,3,3000000
3,129.6,1450000,3,2800000
4,110.0,1150000,2,1600000
5,105.0,1000000,2,1500000


In [6]:
test_df = pd.DataFrame(data=test_data, columns=columns)
test_df

,size,est_price,rooms,price
0,125.0,1300000,3,None


In [11]:
X_train = train_df.drop(columns='price')
X_train

y_train = train_df[['price']]
y_train

,price
0,1200000
1,1900000
2,3000000
3,2800000
4,1600000
5,1500000


In [12]:
X_test = test_df.drop(columns='price')
X_test

y_test = test_df[['price']]
y_test

,price
0,None


In [ ]:
# scaler, ysc = StandardScaler(), StandardScaler()
# Xtr = scaler.fit_transform(X_train) # fit on TRAIN only
# Xte = scaler.transform(X_test)

# ytr = ysc.fit_transform(y_train) # scaled y -> used for training
# yte = y_test.to_numpy(float) # raw y -> used for scoring

In [34]:
Xtr = X_train.to_numpy()
ytr = y_train.to_numpy()

Xte = X_test.to_numpy()
yte = y_test.to_numpy()

In [35]:
Xtr

array([[9.600e+01, 9.000e+05, 2.000e+00],
       [1.180e+02, 1.250e+06, 2.000e+00],
       [1.412e+02, 1.500e+06, 3.000e+00],
       [1.296e+02, 1.450e+06, 3.000e+00],
       [1.100e+02, 1.150e+06, 2.000e+00],
       [1.050e+02, 1.000e+06, 2.000e+00]])

In [36]:
ytr

array([[1200000],
       [1900000],
       [3000000],
       [2800000],
       [1600000],
       [1500000]])

In [37]:
# losses: return (loss, gradient) -> used for training
def mse(y, p):
	return np.mean((p - y) ** 2), 2 * (p - y) / y.size

def softmax_ce(y, logits): # for classification, y is one-hot
	e = np.exp(logits - logits.max(1, keepdims=True)); p = e / e.sum(1, keepdims=True)
	return -np.mean(np.sum(y * np.log(p + 1e-12), 1)), (p - y) / len(y)

# metrics: return a score only -> used for evaluation
def rmse(y, p):
	return np.sqrt(np.mean((y - p) ** 2))

def r2(y, p):
	return 1 - ((y - p) ** 2).sum() / ((y - y.mean()) ** 2).sum()

In [17]:
class Model:
	def __init__(self, n_in, n_hidden, n_out, seed=0):
		rng = np.random.default_rng(seed)
		self.W1 = rng.normal(0, 0.1, (n_in, n_hidden))
		self.b1 = np.zeros(n_hidden)
		self.W2 = rng.normal(0, 0.1, (n_hidden, n_out))
		self.b2 = np.zeros(n_out)

	def forward(self, X):
		self.X = X
		self.z1 = X @ self.W1 + self.b1
		self.a1 = np.maximum(0, self.z1) # ReLU
		return self.a1 @ self.W2 + self.b2

	predict = forward

	def backward(self, d): # d = dLoss/dOutput
		self.dW2 = self.a1.T @ d
		self.db2 = d.sum(0)
		d1 = (d @ self.W2.T) * (self.z1 > 0) # back through W2 and ReLU
		self.dW1 = self.X.T @ d1
		self.db1 = d1.sum(0)

	def step(self, lr):
		self.W1 -= lr * self.dW1
		self.b1 -= lr * self.db1
		self.W2 -= lr * self.dW2
		self.b2 -= lr * self.db2

In [38]:
model = Model(n_in=3, n_hidden=20, n_out=1, seed=42)
lr, epochs = 0.05, 500
history = []

for epoch in range(epochs):
	y_hat = model.forward(Xtr) # 1. forward
	loss, gradient = mse(ytr, y_hat) # 2. loss + gradient
	model.backward(gradient) # 3. backward
	model.step(lr) # 4. update weights
	history.append(loss)
	if epoch % 100 == 0:
		print(f"epoch {epoch:4d}  loss {loss:.4f}")

epoch    0  loss 4550386333947.6006
epoch  100  loss 14220586642455045338775026584113614948401152.0000
epoch  200  loss 10032736373400759988042229618835456.0000
epoch  300  loss 7078174879063059661324288.0000
epoch  400  loss 4994158371668353.0000


In [ ]:
# y_pred_train = ysc.inverse_transform(model.predict(Xtr)) # back to original units
# y_pred_test = ysc.inverse_transform(model.predict(Xte))
# y_true_train = ysc.inverse_transform(ytr) # ytr is scaled, unscale it to compare

# print(f"Train  RMSE: {rmse(y_true_train, y_pred_train):.2f}  R2: {r2(y_true_train, y_pred_train):.3f}")
# print(f"Test   RMSE: {rmse(yte, y_pred_test):.2f}  R2: {r2(yte, y_pred_test):.3f}")

Train  RMSE: 29388.12  R2: 0.998
Test   RMSE: nan  R2: nan


In [40]:
y_pred_train = model.predict(Xtr)
y_pred_test = model.predict(Xte)
y_true_train = ytr

print(f"Train  RMSE: {rmse(y_true_train, y_pred_train):.2f}  R2: {r2(y_true_train, y_pred_train):.3f}")
# print(f"Test   RMSE: {rmse(yte, y_pred_test):.2f}  R2: {r2(yte, y_pred_test):.3f}")

Train  RMSE: 670823.02  R2: -0.000


In [41]:
y_pred_test

array([[1998123.00752245]])